# 01 · Preprocesamiento y normalización de texto

**Taller de una hora.** Trabajamos sobre un corpus externo de reseñas de producto en
español, para que todos los equipos tengan datos desde el primer minuto. Cuando tengáis
vuestro corpus, repetiréis el mismo procedimiento cambiando una línea.

Al terminar tenéis que haber tomado **cuatro decisiones** y haberlas anotado en
`docs/bitacora.md` **con el motivo y con un ejemplo que lo respalde**:

1. Qué tokenizador usáis.
2. Si pasáis a minúsculas, y en qué momento.
3. Qué hacéis con las stopwords.
4. Stemming o lematización.

La rúbrica del Hito 1 no pide el preprocesamiento *aplicado*: pide el preprocesamiento
**aplicado y justificado**. Ejecutar las celdas es la mitad del trabajo.

---

### El corpus

Reseñas de producto de Amazon en español, 5.000 documentos repartidos a partes iguales
entre las cinco puntuaciones (1 a 5 estrellas). Proviene del *Multilingual Amazon Reviews
Corpus*, en la copia publicada en Hugging Face como `mteb/amazon_reviews_multi`.

> La ficha del dataset no declara licencia explícita. **Se usa aquí solo con fines docentes.**
> No lo redistribuyáis: el notebook lo descarga a `data/raw/`, que `.gitignore` excluye, así
> que nunca entra en el repositorio. Es la misma regla que aplicaréis a vuestros datos.

## 0 · Preparación

Lanzad esta celda en cuanto abráis el notebook: tarda un par de minutos la primera vez.

In [ ]:
%pip install -q nltk spacy pandas pyarrow

import importlib.util, subprocess, sys

# El modelo de espanol no viene con spacy: hay que descargarlo una vez.
if importlib.util.find_spec('es_core_news_sm') is None:
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'es_core_news_sm'], check=True)

import nltk
for recurso in ['punkt_tab', 'stopwords']:
    nltk.download(recurso, quiet=True)

print('entorno listo')

## 1 · Descargar el corpus

Se guarda en `data/raw/`. Esa carpeta **no se versiona**: los datos crudos no van al
repositorio, ni por tamaño ni porque pueden contener información de personas reales.

In [ ]:
import pandas as pd
from pathlib import Path

URL = ('https://huggingface.co/api/datasets/mteb/amazon_reviews_multi/parquet/es/test/0.parquet')

raiz = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
cache = raiz / 'data' / 'raw' / 'resenas_es.parquet'
cache.parent.mkdir(parents=True, exist_ok=True)

if cache.exists():
    df = pd.read_parquet(cache)
    print('corpus leido de la copia local')
else:
    df = pd.read_parquet(URL)
    df.to_parquet(cache)
    print('corpus descargado y guardado en data/raw/')

df = df.rename(columns={'label': 'estrellas'})
df['estrellas'] = df['estrellas'].astype(int) + 1   # 0-4 en el origen -> 1-5 estrellas
df = df[['id', 'estrellas', 'text']]
print(f'{len(df)} resenas')
df.head(3)

## 2 · Mirar el corpus antes de tocarlo

Primera regla del preprocesamiento: **mirar los datos**. Longitudes, duplicados, vacíos,
reparto de clases y caracteres raros. Lo que descubráis aquí condiciona todo lo demás.

In [ ]:
s = df['text'].astype(str)

print(f'documentos       : {len(s)}')
print(f'vacios           : {(s.str.strip() == chr(34)+chr(34)).sum()}')
print(f'duplicados       : {s.duplicated().sum()}')
print(f'longitud (car.)  : media {s.str.len().mean():.0f} | min {s.str.len().min()} | max {s.str.len().max()}')
print(f'longitud (palab.): media {s.str.split().str.len().mean():.1f}')
print()
print('resenas por puntuacion:')
print(df['estrellas'].value_counts().sort_index().to_string())

In [ ]:
import re

# Cada fenomeno lleva su patron y si distingue mayusculas de minusculas.
# Importa: '\\bno\\b' distinguiendo mayusculas NO encuentra 'No me llego',
# y el recuento saldria casi a la mitad sin que nada avise.
fenomenos = {
    'con emoji'              : (r'[\U0001F300-\U0001FAFF\u2600-\u27BF]', 0),
    'con apertura de interr.': ('¿', 0),
    'con MAYUSCULAS (4+)'    : (r'\b[A-ZÁÉÍÓÚÑ]{4,}\b', 0),
    'con cifras'             : (r'\d', 0),
    "con 'no'"               : (r'\bno\b', re.IGNORECASE),
    "con 'nunca'"            : (r'\bnunca\b', re.IGNORECASE),
    'con letra repetida 3x'  : (r'(\w)\1\1', re.IGNORECASE),
}

def docs_con(serie, patron, flags=0):
    """En cuantos documentos de la serie aparece el patron.

    Se usa el modulo re de Python y no serie.str.contains porque pandas puede
    delegar en otro motor de expresiones regulares segun como esten
    almacenadas las cadenas, y ese motor no admite los mismos patrones.
    """
    rx = re.compile(patron, flags)
    return int(serie.map(lambda x: bool(rx.search(x))).sum())

base = {}
for nombre, (patron, flags) in fenomenos.items():
    base[nombre] = docs_con(s, patron, flags)
    print(f'  {nombre:26} {base[nombre]:5}  ({100*base[nombre]/len(s):.1f}%)')

Guardad estos números: son la línea base. Al final del notebook comprobaréis cuántos de
estos fenómenos han sobrevivido a vuestro pipeline.

## 3 · La estructura del documento

Antes de tokenizar, una decisión que casi nadie ve. Mirad un documento completo:

In [ ]:
print(repr(s.iloc[0]))

Cada reseña es **título y cuerpo, separados por un salto de línea doble**. No es un texto
plano: es un campo con estructura interna.

Y eso importa, porque el título concentra la valoración (*«ESTAFA EN EL ENVÍO»*,
*«La comida perfecta para mi gata»*) mientras el cuerpo da el detalle. Si los juntáis sin
pensarlo, perdéis esa distinción; si los separáis, tenéis dos señales distintas.

**No hay una respuesta correcta**, pero sí hay que decidirlo y que quede escrito.

In [ ]:
partes = s.str.split('\n\n', n=1)
df['titulo'] = partes.str[0]
df['cuerpo'] = partes.str[1].fillna('')

print(f"documentos con titulo y cuerpo: {(df['cuerpo'] != '').sum()} de {len(df)}")
print(f"longitud media del titulo     : {df['titulo'].str.len().mean():.0f} caracteres")
print(f"longitud media del cuerpo     : {df['cuerpo'].str.len().mean():.0f} caracteres")
print()
df[['estrellas', 'titulo']].head(6)

## 4 · Tokenizar · **decisión 1**

Comparad las dos herramientas más usadas sobre una reseña que contenga una pregunta.

In [ ]:
from nltk.tokenize import word_tokenize
import spacy

nlp = spacy.load('es_core_news_sm')

con_pregunta = s[s.str.contains('\u00bf', regex=False)]
ejemplo = con_pregunta.iloc[0][:120]

print('TEXTO :', ejemplo)
print()
print('NLTK  :', word_tokenize(ejemplo, language='spanish')[:18])
print()
print('spaCy :', [t.text for t in nlp(ejemplo)][:18])

**Fijaos en el signo de apertura.** NLTK devuelve `'¿En'` como un único token; spaCy separa
`'¿'` y `'En'`.

No es cosmético: si `¿En` queda pegado, esa forma no coincidirá con `en` en ningún recuento
ni en ningún vocabulario. Habéis creado un token que no existe. Y en español **toda**
pregunta empieza con `¿`.

NLTK no está roto: está diseñado para el inglés, que no tiene signos de apertura. Ese es el
punto que hay que llevarse — **las herramientas por defecto asumen inglés**, y trabajar en
español obliga a comprobarlo todo. Lo que acaba de pasar no ha dado ningún error: ha
devuelto un resultado plausible y equivocado.

In [ ]:
# A cuantos documentos del corpus les afecta
afectados = s.str.contains('\u00bf', regex=False).sum()
print(f'documentos con apertura de interrogacion: {afectados}')

tok_nltk = set()
for texto in con_pregunta.head(40):
    tok_nltk |= {t for t in word_tokenize(texto, language='spanish') if t.startswith('\u00bf')}
print(f'tokens fantasma que genera NLTK en esos documentos: {len(tok_nltk)}')
print(sorted(tok_nltk)[:12])

> **Decisión 1.** Anotad qué tokenizador usáis y el ejemplo concreto que os hizo decidirlo.

## 5 · Minúsculas y el orden de los pasos · **decisión 2**

Pasar a minúsculas parece inocuo. Comprobadlo sobre una reseña con mayúsculas enfáticas,
que en este corpus hay 253.

In [ ]:
rx_mayus = re.compile(r'\b[A-ZÁÉÍÓÚÑ]{4,}\b')
con_mayus = s[s.map(lambda x: bool(rx_mayus.search(x)))]
frase = con_mayus.iloc[0].split('\n\n')[0]
print('TITULO:', frase)
print()

for etiqueta, texto in [('TAL CUAL', frase), ('EN MINUSCULAS', frase.lower())]:
    print(f'--- {etiqueta} ---')
    print('  %-14s %-14s %-8s' % ('TOKEN', 'LEMA', 'POS'))
    for t in nlp(texto):
        if not t.is_punct:
            print('  %-14s %-14s %-8s' % (t.text, t.lemma_, t.pos_))
    print()

Comparad las dos tablas palabra por palabra. La misma palabra en mayúsculas recibe **otra
categoría gramatical y otro lema**: `ESTAFA` sale etiquetada como adjetivo y su lema es
`estafo`, que no existe en español; en minúsculas sale como nombre y su lema es `estafa`.
En otros casos la palabra en mayúsculas se toma por un nombre propio (`PROPN`) y entonces no
se lematiza en absoluto.

El motivo es que la mayúscula **es una de las señales que usa el etiquetador** para decidir
la categoría. Si se la dais donde no toca, se equivoca, y el lema se equivoca con él.

Dos consecuencias:

1. **Lematizar antes o después de bajar a minúsculas da corpus distintos.** El orden no es
   un detalle de implementación: es una decisión metodológica.
2. En reseñas, **las mayúsculas son énfasis**. `ESTAFA` en mayúsculas dice algo sobre la
   intensidad. Si lo primero que hacéis es bajar todo a minúsculas, tiráis esa señal antes
   de haberla mirado.

> **Decisión 2.** Si vuestro proyecto va de sentimiento, conservad una columna con el texto
> original y, si os interesa, una variable con la proporción de mayúsculas. Cuesta cero y os
> deja volver atrás.

In [ ]:
# El enfasis en mayusculas, tiene relacion con la puntuacion?
df['grita'] = s.map(lambda x: bool(rx_mayus.search(x)))
print('% de resenas con alguna palabra en mayusculas, por puntuacion:')
print(df.groupby('estrellas')['grita'].mean().mul(100).round(1).to_string())

El porcentaje baja de forma sostenida de 1 a 5 estrellas: **escribir en mayúsculas es más
frecuente cuanto peor es la valoración.** Es una señal real, medida sobre el corpus, y está
en el formato del texto, no en las palabras.

Si bajáis todo a minúsculas sin guardar nada, esa señal se va. Y fijaos en lo que acabáis de
hacer: habéis construido una variable útil (`grita`) **a partir de algo que el
preprocesamiento iba a destruir**. Eso es el tipo de hallazgo que se valora en el EDA del
Hito 1.

## 6 · Stopwords · **decisión 3**

Las stopwords son las palabras funcionales, las que aparecen en todos los documentos y en
teoría no distinguen nada. Se suelen quitar sin pensar. Vamos a pensarlo.

In [ ]:
from nltk.corpus import stopwords

sw_nltk = set(stopwords.words('spanish'))
doc = nlp(s.iloc[1].lower())

print('%-16s %-11s %-11s' % ('TOKEN', 'NLTK', 'spaCy'))
for t in doc:
    if t.is_punct or t.is_space:
        continue
    print('%-16s %-11s %-11s' % (t.text,
          'stopword' if t.text in sw_nltk else '-',
          'stopword' if t.is_stop else '-'))

Primer hallazgo: **las dos listas no coinciden.** Vienen de las dos librerías más usadas y
no están de acuerdo. No existe *la* lista correcta de stopwords: existen listas, hechas por
alguien, con criterios que no os han contado. Usar una sin abrirla es delegar una decisión
de vuestro proyecto en un fichero que no habéis leído.

In [ ]:
negaciones = ['no', 'ni', 'nada', 'sin', 'nunca', 'tampoco', 'jam\u00e1s', 'nadie', 'ning\u00fan']

print(f'la lista de NLTK tiene {len(sw_nltk)} palabras')
print()
print('%-12s %-18s %s' % ('PARTICULA', 'EN LA LISTA?', 'RESENAS QUE LA CONTIENEN'))
for w in negaciones:
    n = s.str.contains(rf'\b{w}\b', regex=True, case=False).sum()
    marca = 'SI -> se borra' if w in sw_nltk else 'no -> se queda'
    print('%-12s %-18s %d' % (w, marca, n))

Segundo hallazgo, y este es grave para cualquier proyecto de sentimiento.

La lista contiene `no`, `ni`, `nada` y `sin`, pero **no** contiene `nunca`, `tampoco` ni
`jamás`. Filtrando a ciegas:

- *«**no** me gustó **nada**»* se queda en *«gustó»* — **sentido invertido**;
- *«**nunca** volveré a comprar»* se queda intacto.

O sea que el filtro no solo destruye la negación: **la destruye de forma incoherente**, en
unas reseñas sí y en otras no. Vuestro modelo aprendería de un corpus en el que parte de las
negaciones han desaparecido y parte no, sin ningún criterio.

In [ ]:
# Cuantas resenas del corpus quedan con el sentido alterado
borra = [w for w in negaciones if w in sw_nltk]
conserva = [w for w in negaciones if w not in sw_nltk]
afecta = s.str.contains(r'\b(?:' + '|'.join(borra) + r')\b', regex=True, case=False).sum()
print(f'particulas que la lista borra   : {borra}')
print(f'particulas que la lista conserva: {conserva}')
print()
print(f'resenas con alguna particula borrada: {afecta} de {len(s)} ({100*afecta/len(s):.1f}%)')
print('En ese porcentaje del corpus, el filtro cambia lo que el texto dice.')

In [ ]:
# Una lista propia: la de NLTK menos las negaciones
NEGACIONES = {'no', 'ni', 'nada', 'sin', 'nunca', 'tampoco', 'jam\u00e1s', 'nadie',
              'ninguno', 'ning\u00fan', 'ninguna', 'apenas'}
sw_propia = sw_nltk - NEGACIONES

print(f'lista original: {len(sw_nltk)}  ->  lista propia: {len(sw_propia)}')
print()
prueba = 'no me gust\u00f3 nada el env\u00edo y encima nunca avisaron'
print('texto          :', prueba)
print('con la original:', [w for w in prueba.split() if w not in sw_nltk])
print('con la propia  :', [w for w in prueba.split() if w not in sw_propia])

> **Decisión 3.** ¿Filtráis stopwords? Si vuestro proyecto tiene que ver con sentimiento, lo
> razonable es partir de la lista y **retirar de ella las partículas de negación**. Y si no
> filtráis, decidlo también: TF-IDF ya penaliza por sí solo las palabras que aparecen en
> todos los documentos, como veréis la semana que viene.

## 7 · Stemming frente a lematización · **decisión 4**

Las dos técnicas agrupan formas de una misma palabra, pero no son equivalentes.

In [ ]:
from nltk.stem import SnowballStemmer

stemmer = SnowballStemmer('spanish')
muestra = ' '.join(s.head(60).tolist()).lower()

filas = []
for t in nlp(muestra):
    if t.is_punct or t.is_space or t.like_num or len(t.text) < 3:
        continue
    filas.append({'token': t.text, 'stem': stemmer.stem(t.text),
                  'lema': t.lemma_, 'pos': t.pos_})

comp = pd.DataFrame(filas).drop_duplicates('token').reset_index(drop=True)
difieren = comp[comp['stem'] != comp['lema']]
print(f'{len(comp)} tokens distintos | difieren en {len(difieren)} ({100*len(difieren)/len(comp):.0f}%)')
difieren.head(20)

- El **stemmer** corta por reglas. Es rápido, no sabe nada de español y devuelve formas que
  no son palabras (`envío`→`envi`, `producto`→`product`). Agrupa bien, se lee mal.
- El **lematizador** consulta un modelo de la lengua y devuelve la forma de diccionario
  (`llegó`→`llegar`). Da palabras reales, es más lento y **se equivoca**.

El criterio para elegir es más simple de lo que parece: **¿el resultado lo va a leer
alguien?** Si en diciembre vais a proyectar una lista de causas de queja, necesitáis lemas,
porque `envi` no se puede poner en una diapositiva. Si solo vais a contar para alimentar un
modelo, el stem basta y es más rápido.

Ahora comprobad dónde falla el lematizador. En este corpus hay 617 reseñas con un pronombre
pegado al verbo (`devolverlo`, `comprarlo`, `pedírmelo`):

In [ ]:
clitico = re.compile(r'\w+(?:melo|selo|rlo|rla|rme|rnos)$')
vistos, n = set(), 0
print('%-16s %-16s %-8s' % ('TOKEN', 'LEMA DE spaCy', 'POS'))
for t in nlp(' '.join(s.head(150).tolist()).lower()):
    if clitico.match(t.text) and t.text not in vistos:
        vistos.add(t.text)
        print('%-16s %-16s %-8s' % (t.text, t.lemma_, t.pos_))
        n += 1
        if n >= 12:
            break

Mirad la columna del lema: `devolverlo` no da `devolver`, da **`'devolver él'`** — un lema
con un espacio dentro. spaCy sí desdobla el pronombre, pero lo hace *dentro del lema*, sin
partir el token.

Y eso rompe algo silenciosamente: si metéis ese lema en vuestro vocabulario, tenéis una
«palabra» que en realidad son dos, y no va a coincidir con `devolver` en ningún recuento.
Hay un caso peor: algunas formas devuelven el lema con un **espacio final** (`ofrecéis` →
`'ofreceis '`), que a la vista es indistinguible de la palabra normal.

Comprobad cuántos hay en el corpus:

In [ ]:
multi = {}
for t in nlp(' '.join(s.head(300).tolist()).lower()):
    if t.is_space or t.is_punct:
        continue
    if t.lemma_ != t.lemma_.strip() or ' ' in t.lemma_.strip():
        multi.setdefault(t.lemma_, t.text)

print(f'lemas con espacio en 300 documentos: {len(multi)}')
for lema, tok in list(multi.items())[:8]:
    print(f'  {tok:16} -> {lema!r}')

> **Decisión 4.** Stem o lema. Y si elegís lema, **qué hacéis con los lemas que traen
> espacios**: partirlos en dos tokens, quedarse con el verbo, o dejarlos. El pipeline de la
> sección 8 los parte; es una decisión, no un arreglo evidente.

### Un efecto del orden que no se ve venir

Queda un problema, y es el más instructivo de todo el taller. Comprobad si las formas de
`ser` y su lema están en la lista de stopwords:

In [ ]:
for lema, formas in [('ser',   ['es', 'son', 'era', 'fue', 'soy', 'sea']),
                     ('haber', ['ha', 'han', 'he', 'hay']),
                     ('tener', ['tiene', 'tengo']),
                     ('estar', ['esta', 'estoy'])]:
    dentro = [f for f in formas if f in sw_nltk]
    print(f"  lema '{lema:6}' {'SI esta' if lema in sw_nltk else 'NO esta':8} en la lista "
          f'| formas suyas que si estan: {dentro}')

Ahí está. La lista contiene `es`, `son`, `era`, `fue`… pero **no contiene `ser`**. Lo mismo
con `haber` y `tener`. En cambio `estar` sí está: la lista es incoherente consigo misma.

Las consecuencias son concretas:

- Si **filtráis y luego lematizáis**, `es` y `son` se eliminan por estar en la lista.
- Si **lematizáis y luego filtráis** —que es lo que hace el pipeline de la sección 8—,
  `es` y `son` se convierten antes en `ser`, que no está en la lista, y **sobreviven todos**.

Los mismos dos pasos, en distinto orden, dan corpus distintos. En la sección 9 lo vais a ver
medido: `ser` y `haber` aparecerán entre los términos más frecuentes del corpus «limpio»,
que es justo lo que el filtro pretendía evitar.

**No hay una solución automática.** O filtráis antes de lematizar, o añadís los lemas de los
verbos auxiliares a vuestra lista. Lo que no vale es no darse cuenta.

## 8 · Montar el pipeline con vuestras decisiones

Estas siete líneas **son** vuestro preprocesamiento: es lo que tenéis que poder explicar en
el Hito 1. Cambiadlas y volved a ejecutar para ver el efecto.

In [ ]:
TOKENIZADOR          = 'spaCy es_core_news_sm'   # decision 1
USAR_MINUSCULAS      = True                      # decision 2
FILTRAR_STOPWORDS    = True                      # decision 3
CONSERVAR_NEGACIONES = True                      #   ''
MODO                 = 'lema'                    # decision 4: 'lema' o 'stem'
FILTRAR_ANTES        = False                     # True: filtrar la forma original,
                                                 # antes de lematizar (ver seccion 7)
MIN_LONGITUD         = 2
CAMPO                = 'text'                    # 'text', 'titulo' o 'cuerpo'

lista_sw = sw_propia if CONSERVAR_NEGACIONES else sw_nltk

def preprocesar(doc):
    salida = []
    for t in doc:
        if t.is_punct or t.is_space or t.like_url or t.like_email:
            continue
        # Filtrar aqui usa la forma tal como aparece en el texto.
        if FILTRAR_STOPWORDS and FILTRAR_ANTES and t.text.lower() in lista_sw:
            continue
        pieza = (t.lemma_ if MODO == 'lema' else stemmer.stem(t.text)).lower()
        # Un lema puede traer espacios ('devolver el', 'ofreceis '): se parte.
        for trozo in pieza.split():
            if len(trozo) < MIN_LONGITUD:
                continue
            if FILTRAR_STOPWORDS and not FILTRAR_ANTES and trozo in lista_sw:
                continue
            salida.append(trozo)
    return salida

textos = df[CAMPO].astype(str)
if USAR_MINUSCULAS:
    textos = textos.str.lower()

# nlp.pipe es mucho mas rapido que llamar a nlp() documento a documento
df['tokens'] = [preprocesar(d) for d in nlp.pipe(textos.tolist(), batch_size=200)]
df['n_tokens'] = df['tokens'].apply(len)

print(f'procesados {len(df)} documentos')
df[['estrellas', 'titulo', 'tokens']].head(5)

## 9 · Qué ha cambiado y qué se ha perdido

Nunca deis un preprocesamiento por bueno sin medir su efecto.

In [ ]:
from collections import Counter

crudo = Counter(' '.join(s).lower().split())
limpio = Counter(tok for lista in df['tokens'] for tok in lista)

print(f'vocabulario : {len(crudo):6} -> {len(limpio):6}')
print(f'tokens      : {sum(crudo.values()):6} -> {sum(limpio.values()):6}')
print()
print('20 terminos mas frecuentes tras el preprocesamiento:')
for w, n in limpio.most_common(20):
    print(f'  {w:<16} {n}')

**Comprobación de sentido.** Mirad esa lista de veinte y buscad dos cosas.

La primera: **`ser` y `haber` en los primeros puestos**, con miles de apariciones. Son
exactamente los verbos auxiliares que el filtro de stopwords pretendía eliminar, y están ahí
por el efecto del orden de la sección 7: al lematizar antes de filtrar, `es` y `son` se
convierten en `ser`, que no figura en la lista.

Probadlo: poned `FILTRAR_ANTES = True` en la sección 8, ejecutad de nuevo esa celda y esta, y
comparad las dos listas de veinte términos. Esa es la magnitud del efecto de una decisión
que ni siquiera sabíais que estabais tomando.

La segunda: palabras que en este dominio **no distinguen nada** — `producto`, `comprar`,
`calidad` aparecen en reseñas buenas y malas por igual. Ninguna lista genérica las contiene,
porque solo sobran en este corpus. Las stopwords específicas de un dominio hay que añadirlas
mirando los datos, y esto es cómo se hace.

In [ ]:
# Los fenomenos de la seccion 2, han sobrevivido?
# Se cuentan DOCUMENTOS en los dos casos, para que las cifras sean comparables.
limpio_txt = df['tokens'].apply(' '.join)

print('%-26s %-9s %-9s %s' % ('FENOMENO', 'ANTES', 'DESPUES', 'SUPERVIVENCIA'))
for nombre, (patron, flags) in fenomenos.items():
    antes = base[nombre]
    despues = docs_con(limpio_txt, patron, flags)
    pct = f'{100*despues/antes:.0f}%' if antes else '-'
    print('%-26s %-9s %-9s %s' % (nombre, antes, despues, pct))

In [ ]:
# Un documento, antes y despues
i = 0
print('ANTES  :', s.iloc[i][:200])
print()
print('DESPUES:', df['tokens'].iloc[i])

Leed la tabla fila por fila.

La de mayúsculas baja a cero **por construcción**: habéis pasado todo a minúsculas, así que
era de esperar. No es un hallazgo.

Los **emojis sí lo son**: han desaparecido de los veinte documentos que los tenían, y **no
los ha quitado ninguna lista de stopwords**. Los ha quitado `MIN_LONGITUD`, porque un emoji
es un solo carácter. Igual con buena parte de las cifras. Nadie decidió eso; fue el efecto
colateral de una línea que parecía inocua.

Y fijaos en qué se pierde. En un proyecto de sentimiento, un emoji puede ser la señal más
clara del documento; si os interesan los plazos de entrega, `3 días` era el dato.

Es la lección del taller: **un paso de preprocesamiento que no habéis justificado está
borrando datos sin que lo sepáis.** Decidid a propósito qué hacéis con números y emojis:
conservarlos, sustituirlos por una etiqueta (`<NUM>`, `<EMOJI>`) o eliminarlos.

## 10 · Para qué servía todo esto

El preprocesamiento no es el objetivo: es lo que permite que un recuento signifique algo.
Comparad los términos característicos de las reseñas de 1 estrella y de 5.

In [ ]:
frec = {}
for e in [1, 5]:
    c = Counter(tok for lista in df[df.estrellas == e]['tokens'] for tok in lista)
    total = sum(c.values())
    frec[e] = {w: n / total for w, n in c.items()}

comunes = set(frec[1]) & set(frec[5])
ratio = sorted(((frec[1][w] / frec[5][w], w) for w in comunes if limpio[w] >= 20),
               reverse=True)

print('Mas caracteristicos de 1 ESTRELLA:')
for r, w in ratio[:12]:
    print(f'  {w:<18} x{r:.1f}')
print()
print('Mas caracteristicos de 5 ESTRELLAS:')
for r, w in ratio[-12:][::-1]:
    print(f'  {w:<18} x{1/r:.1f}')

Esto ya es un resultado de marketing: las palabras que separan a un cliente satisfecho de
uno enfadado, y en qué proporción. Es exactamente el tipo de análisis que pide el **EDA
textual del Hito 1**, y la base de lo que haremos en la semana 5 con análisis de sentimiento.

**Y ahora la parte importante:** volved a la sección 8, cambiad `CONSERVAR_NEGACIONES` a
`False`, ejecutad esa celda y volved a ejecutar esta. Mirad qué pasa con las palabras
características de 1 estrella. Eso es el coste real de una decisión que parecía técnica.

## 11 · Guardar y documentar

In [ ]:
destino = raiz / 'data' / 'clean' / 'corpus_preprocesado.csv'
destino.parent.mkdir(parents=True, exist_ok=True)
guardar = df[['id', 'estrellas', 'titulo', 'cuerpo', 'tokens', 'n_tokens']].copy()
guardar['tokens'] = guardar['tokens'].apply(' '.join)
guardar.to_csv(destino, index=False, encoding='utf-8')
print('guardado en', destino.relative_to(raiz), f'({destino.stat().st_size/1024:.0f} KB)')

print()

# Se imprime la entrada de la bitacora ya rellenada con vuestra configuracion.
orden = 'antes de lematizar' if FILTRAR_ANTES else 'despues de lematizar'
negs  = 'si' if CONSERVAR_NEGACIONES else 'NO'
filas = [
    ('1 · Tokenizador', TOKENIZADOR),
    ('2 · Minusculas', f'{"si" if USAR_MINUSCULAS else "no"}, filtrando {orden}'),
    ('3 · Stopwords', f'{"si" if FILTRAR_STOPWORDS else "no"} ({len(lista_sw)} palabras), '
                     f'negaciones conservadas: {negs}'),
    ('4 · Normalizacion', MODO),
]

print('=' * 78)
print('COPIAD DESDE AQUI A docs/bitacora.md, entrada "Preprocesamiento del corpus"')
print('=' * 78)
print()
print(f'Corpus: mteb/amazon_reviews_multi (es) - {len(df)} resenas - campo: {CAMPO}')
print(f'Efecto medido: vocabulario {len(crudo)} -> {len(limpio)} terminos; '
      f'{sum(crudo.values())} -> {sum(limpio.values())} tokens')
print()
print('| Decision | Que elegimos | Por que | Ejemplo del corpus |')
print('|---|---|---|---|')
for nombre, valor in filas:
    print(f'| {nombre} | {valor} | RELLENAD | RELLENAD |')
print()
print('Limitaciones que asumimos: RELLENAD')
print()
print('-' * 78)
print('Las dos columnas RELLENAD son las que se evaluan. Sin ellas, la entrada')
print('no vale: la rubrica pide el preprocesamiento justificado, no solo aplicado.')

---

## Antes de cerrar

- [ ] `data/clean/corpus_preprocesado.csv` existe.
- [ ] Las **cuatro decisiones** están en `docs/bitacora.md`, con el motivo.
- [ ] Para cada decisión hay un ejemplo concreto que la respalda.
- [ ] Habéis ejecutado la sección 10 con `CONSERVAR_NEGACIONES` en `True` y en `False`, y
      sabéis explicar la diferencia.
- [ ] El notebook está confirmado en el repositorio.

`data/raw/` y `data/clean/` **no se suben**: los excluye el `.gitignore`. Lo que se sube es
el notebook y la bitácora — con eso cualquiera reproduce el resultado.

## Cuando tengáis vuestro corpus

Cambiad la sección 1 para leer vuestro fichero de `data/raw/` y ajustad `CAMPO` en la
sección 8. El resto del notebook funciona igual. Las cuatro decisiones habrá que
**revisarlas** sobre vuestros datos: la lista de stopwords que sirve para reseñas de Amazon
no tiene por qué servir para titulares de prensa o para mensajes de soporte.